In [23]:
import pandas as pd
import numpy as np
import os
import math
import seaborn as sns
import matplotlib.pyplot as plt
import librosa
import copy
import librosa.display
from IPython.display import Audio
import csv
import warnings
warnings.filterwarnings('ignore')
import pywt
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

In [35]:
# Define CNN-LSTM model
class CNN_LSTM_Model(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes):
        super(CNN_LSTM_Model, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm1d(32)  # Add batch normalization layer
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm1d(64)  # Add batch normalization layer
        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)
        self.lstm = nn.LSTM(64, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=1)
        
    def forward(self, x):
        x = x.unsqueeze(1)  # Add channel dimension
        x = self.conv1(x)
        x = self.bn1(x)  # Apply batch normalization
        x = self.relu(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = self.bn2(x)  # Apply batch normalization
        x = self.relu(x)
        x = self.pool(x)
        x = x.permute(0, 2, 1)  # Reshape for LSTM input (batch_size, seq_len, input_size)
        lstm_out, _ = self.lstm(x)
        lstm_out = lstm_out[:, -1, :]  # Take only the output of the last time step
        out = self.fc(lstm_out)
        out = self.softmax(out)
        return out
# Define custom dataset
class SpeechDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [3]:

def predict_emotion(model, extract_features):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    with torch.no_grad():
        inputs = torch.tensor(extract_features, dtype=torch.float32).unsqueeze(0).to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        return predicted.item()

In [4]:
# Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Define a dictionary mapping wavelet levels to their corresponding feature sizes
wavelet_feature_sizes = {
    1: 80,
    2: 120,
    3: 160,
    4: 200,
    5: 240
}


In [37]:
input_size = 200
hidden_size = 256
num_layers = 1
num_classes = 6

In [38]:
model = CNN_LSTM_Model(input_size, hidden_size, num_layers, num_classes).to(device)
model.load_state_dict(torch.load(r"C:\Users\Tharunraj_S\Desktop\Major project\codes\Pytorch\DWT and MFCC\CNN_LSTM_Model\savee\N_MFCC40\level 1 wavelet\500 Epochs\LSTM_model_(best).pth")) 
model.eval()

CNN_LSTM_Model(
  (conv1): Conv1d(1, 32, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv1d(32, 64, kernel_size=(3,), stride=(1,), padding=(1,))
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (lstm): LSTM(64, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=6, bias=True)
  (relu): ReLU()
  (softmax): Softmax(dim=1)
)

In [39]:
def extract_features(filename):
    y, sr = librosa.load(filename, duration=3, offset=0.5)

    global dwt_level
    # Apply DWT
    dwt_level=1
    coeffs = pywt.wavedec(y, 'db1', level=dwt_level) # Adjust wavelet and decomposition level as needed
    mfcc_features = []
    
    for coeff in coeffs:
        mfcc_coeff = np.mean(librosa.feature.mfcc(y=coeff.astype(float), sr=sr, n_mfcc=40).T, axis=0)
        mfcc_features.extend(mfcc_coeff)
    return np.array(mfcc_features)

In [40]:
def live_prediction(model):
    CHUNK = 1024
    FORMAT = pyaudio.paInt16
    CHANNELS = 1
    RATE = 44100
    RECORD_SECONDS = 10  # Adjust the duration of each recording as needed
    
    print(f"Prediction every {RECORD_SECONDS} seconds")
    p = pyaudio.PyAudio()

    try:
        while True:
            print("Listening...")

            stream = p.open(format=FORMAT,
                            channels=CHANNELS,
                            rate=RATE,
                            input=True,
                            frames_per_buffer=CHUNK)

            frames = []

            for i in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
                data = stream.read(CHUNK)
                frames.append(data)

            print("Recording stopped.")

            # Close the audio stream
            stream.stop_stream()
            stream.close()

            # Save audio data to a file for processing (you can skip this step if processing in memory)
            wf = wave.open("temp.wav", 'wb')
            wf.setnchannels(CHANNELS)
            wf.setsampwidth(p.get_sample_size(FORMAT))
            wf.setframerate(RATE)
            wf.writeframes(b''.join(frames))
            wf.close()

            # Extract MFCC features from the recorded audio file
            features = extract_features("temp.wav")

            # Predict emotion label
            predicted_label_index = predict_emotion(model, features)

            emotion_labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'pleasant_surprised', 'sad']
            predicted_emotion = emotion_labels[predicted_label_index]

            print(f'Predicted emotion label: {predicted_emotion}')

    except KeyboardInterrupt:
        print("Stopping...")
        p.terminate()



In [14]:
live_prediction(model)

Prediction every 10 seconds
Listening...
Recording stopped.
Predicted emotion label: fear
Listening...
Recording stopped.
Predicted emotion label: disguist
Listening...
Recording stopped.
Predicted emotion label: disguist
Listening...
Recording stopped.
Predicted emotion label: disguist
Listening...
Recording stopped.
Predicted emotion label: angry
Listening...
Recording stopped.
Predicted emotion label: happy
Listening...
Recording stopped.
Predicted emotion label: disguist
Listening...
Stopping...


In [41]:
features = extract_features(r"C:\Users\Tharunraj_S\Desktop\Major project\Dataset\combined_dataset\angry\YAF_white_angry.wav")

In [42]:
print(features.shape)

(80,)


In [43]:
predicted_label_index = predict_emotion(model, features)
emotion_labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad']
predicted_emotion = emotion_labels[predicted_label_index]
print(f'Predicted emotion label: {predicted_emotion}')

Predicted emotion label: fear
